In [ ]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------



import datetime

import pandas as pd

from pandas import ExcelWriter

from selenium import webdriver


from selenium.webdriver.common.by import By

from time import sleep

import os











    

In [2]:

# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'IS CBI'
print(f"Running {regulatorName} Web Scraping Tool v.1.1")

now=datetime.datetime.now()

filename= f'{regulatorName} SQL Ready {str(now).replace(":",".")[:-7]}.xlsx'

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)





Running IS CBI Web Scraping Tool v.1.1


In [3]:

# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------


def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict




In [4]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict = {regulatorName+' 1' : 'https://cb.is/financial-supervision/regulated-activities/supervised-entities/'}

Typology = {

            regulatorName+" 1": "Supervised entities",


            }



sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

          'Phone - Mother company': [], 'Check': []}





processdate = now.strftime('%Y-%m-%d')



In [5]:

# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

#driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()), options=chromeOptions )

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()



In [6]:

# %%

#------------------------------------------------ Begin_Main ----------------------------------------

for k, reg in enumerate(regdict):

    print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_ ")

    driver.get(regdict[reg])

    sleep(3)

    

    driver.find_element(By.XPATH, "//a[contains(@class,'btn')][normalize-space()='Supervised entities']").click()
    # check_dowload_files(tempfolder, 'xlsx')
    sleep(5)
    

    filePath = os.path.join(tempfolder, os.listdir(tempfolder)[0])

    tempdf = pd.read_excel(filePath)

    os.remove(filePath)
    

    print(reg,tempdf.shape)

    sqldict['Name'].extend(tempdf['Name'])

    sqldict['ListProcessDate'].extend([processdate]*len(tempdf['Name']))

    sqldict['ListName'].extend([Typology[reg]]*len(tempdf['Name']))

    sqldict['ListCode'].extend(['1']*len(tempdf['Name']))

    sqldict['RegCode'].extend(['CBI']*len(tempdf['Name']))

    sqldict['RegCtry'].extend(['IS']*len(tempdf['Name']))

    sqldict['RegulationType'].extend(['Regulated']*len(tempdf['Name']))

    sqldict['Address_1'].extend(tempdf['Address'])

    sqldict['City'].extend(tempdf['City'])
    sqldict['InternalID_1'].extend(tempdf['SSN'])
    sqldict['InternalID_1_type'].extend(['SSN']*len(tempdf['Name']))
    sqldict['Zip'].extend(tempdf['Postal code'])

    sqldict['Typology'].extend(tempdf['Operation'])
    sqldict = bourange_same_length_array(sqldict)


[INFO] : Working 1/1 _(IS CBI 1)_ 
IS CBI 1 (137, 6)


In [7]:
		
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df.to_excel(writer, 'SQL Ready', index=False)

writer.save()
writer.close()
sleep(3)

driver.quit()
    

C:\Users\wuj1\AppData\Local\Temp\2\ipykernel_27052\4202423384.py:5: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [8]:
df.to_excel(filename, index=False)